In [14]:
#resolving path issues for importing alfredgraph.csv
from pathlib import Path
import pandas as pd
import re

path = Path("../data/raw/alfredgraph.csv")

print(path.resolve())
print(path.exists())

raw = pd.read_csv(
    path,
    na_values=["", "."]
)

C:\Users\zakar\Jupyter_Projects\copper_project\data\raw\alfredgraph.csv
True


In [15]:
#separating time-series observation date from data vintage date & debugging.
raw["observation_date"] = pd.to_datetime(raw["observation_date"])

vintage_columns = [
    column 
    for column in raw.columns
    if re.fullmatch(r"PCOPPUSDM_\d{8}", column)
]

print(vintage_columns)

latest_column = max(vintage_columns)

print(raw.columns.tolist())

['PCOPPUSDM_20260605', 'PCOPPUSDM_20260713']
['observation_date', 'PCOPPUSDM_20260605', 'PCOPPUSDM_20260713']


In [16]:
#column subsetting and renaming.
copper = (
    raw[["observation_date", latest_column]]
    .rename(columns={
        "observation_date": "date",
        latest_column: "copper_price_usd_per_tonne"
    })
)

In [17]:
#type conversion, i.e., numeric coercion.
copper["copper_price_usd_per_tonne"] = pd.to_numeric(
    copper["copper_price_usd_per_tonne"],
    errors="coerce"
)

In [18]:
#date normalization
copper["date"] = (
    copper["date"]
    .dt.to_period("M")
    .dt.to_timestamp("M")
)

In [19]:
#data cleaning and ordering.
copper = (
    copper
    .dropna(subset=["copper_price_usd_per_tonne"])
    .sort_values("date")
    .drop_duplicates("date")
)

In [20]:
#metadata enrichment
copper["source"] = "IMF Primary Commodity Prices via ALFRED"
copper["frequency"] = "monthly"
copper["unit"] = "USD per metric tonne"
copper["vintage_date"] = pd.to_datetime(
    latest_column[-8:],
    format="%Y%m%d"
)

In [31]:
#importing the processed data as a .csv file.
copper.to_csv(
    "../data/processed/copper_price_monthly.csv",
    index=False
)

In [21]:
#config file stating assumptions & design decisions
import yaml
with open("../config/assumptions.yaml", "r", encoding="utf-8") as file:
    assumptions = yaml.safe_load(file)

base_currency = assumptions["project"]["base_currency"]
date_convention = assumptions["date_handling"]["standard_date"]

In [22]:
#producing validation record
validation_record = {
    "column_name": "copper_price_usd_per_tonne",
    "observations": len(copper),
    "start_date": copper["date"].min(),
    "end_date": copper["date"].max(),
    "missing_values": copper["copper_price_usd_per_tonne"].isna().sum(),
    "duplicate_dates": copper["date"].duplicated().sum(),
    "minimum": copper["copper_price_usd_per_tonne"].min(),
    "maximum": copper["copper_price_usd_per_tonne"].max(),
}

In [24]:
#checking validation result
pd.Series(validation_record)

column_name        copper_price_usd_per_tonne
observations                              414
start_date                1992-01-31 00:00:00
end_date                  2026-06-30 00:00:00
missing_values                              0
duplicate_dates                             0
minimum                           1377.376087
maximum                          13552.040909
dtype: object

In [25]:
#importing raw dataset for refinery production
import pandas as pd
from pathlib import Path

raw_path = Path(
    "../data/raw/"
    "statista_global_refinery_copper_production_kt_annual_raw.csv"
)

#the file has no formal columns, so standardizing the fields
raw = pd.read_csv(raw_path)

raw.columns = ["year", "refined_copper_production_kt"]

raw.head()

,year,refined_copper_production_kt
0,2000,14793
1,2001,15638
2,2002,15354
3,2003,15272
4,2004,15918


In [30]:
#cleaning the year and production fields
raw["year"] = (
    raw["year"]
    .astype(str)
    .str.replace("*", "", regex=False)
)

raw["year"] = pd.to_numeric(raw["year"], errors="coerce")

raw["refined_copper_production_kt"] = pd.to_numeric(
    raw["refined_copper_production_kt"],
    errors="coerce"
)

In [31]:
#creating a standardized annual date
production = raw.copy()

production["date"] = pd.to_datetime(
    production["year"].astype("Int64").astype(str) + "-12-31"
)

production = production[
    ["date", "refined_copper_production_kt"]
    ].sort_values("date")

In [32]:
#validating the dataset
validation_record = {
    "observations": len(production),
    "start_date": production["date"].min(),
    "end_date": production["date"].max(),
    "missing_dates": production["date"].isna().sum(),
    "missing_values": (
        production["refined_copper_production_kt"].isna().sum()
    ),
    "duplicate_dates": production["date"].duplicated().sum(),
    "minimum_value": (
        production["refined_copper_production_kt"].min()
    ),
    "maximum_value": (
        production["refined_copper_production_kt"].max()
    ),
}

pd.Series(validation_record)

observations                        25
start_date         2000-12-31 00:00:00
end_date           2024-12-31 00:00:00
missing_dates                        0
missing_values                       0
duplicate_dates                      0
minimum_value                    14793
maximum_value                    27486
dtype: object

In [33]:
#ensuring each year from 2000 to 2024 is present
expected_years = set(range(2000, 2025))
actual_years = set(production["date"].dt.year)

missing_years = expected_years - actual_years
extra_years = actual_years - expected_years

print("Missing Years:", missing_years)
print("Extra Years:", extra_years)

Missing Years: set()
Extra Years: set()


In [35]:
#confirming that the production column is numeric
production.dtypes

date                            datetime64[ns]
refined_copper_production_kt             int64
dtype: object

In [36]:
#saving the cleaned series
production.to_csv(
    "../data/processed/"
    "refined_copper_production_annual_clean.csv",
    index=False
)

In [40]:
#updating data dictionary
data_dictionary = pd.read_csv(
    "../data/metadata/data_dictionary.csv"
)

new_entry = {
    "column_name": "refined_copper_production_kt",
    "dataset": "annual copper fundamentals",
    "definition": "Global refined copper production",
    "unit": "thousand metric tonnes",
    "frequency": "annual",
    "data_type": "numeric",
    "date_convention": "calendar year-end",
    "transformation": "Cleaned Statista/ICSG series",
    "source_id": "SRC-002"
}

data_dictionary = pd.concat(
    [data_dictionary, pd.DataFrame([new_entry])],
    ignore_index=True
)

data_dictionary.to_csv(
    "../data/metadata/data_dictionary.csv",
    index=False
)

In [46]:
#updating source register

source_register = pd.read_csv(
    "../data/metadata/source_register.csv"
)

source_entry = {
    "source_id": "SRC-002",
    "variable_or_dataset": "refined_copper_production_kt",
    "provider": "Statista",
    "underlying_source": "International Copper Study Group",
    "series_name": "Global copper refinery production",
    "url": "https://www.statista.com/statistics/254917/total-global-copper-production-since-2006/",
    "frequency": "annual",
    "unit": "thousand metric tonnes",
    "access_date": "2026-08-03",
    "coverage": "2000-2024",
    "source_type": "secondary reproduction"
}

source_register = pd.concat(
    [source_register, pd.DataFrame([source_entry])],
    ignore_index=True
)

source_register.to_csv(
    "../data/metadata/source_register.csv",
    index=False
)

In [52]:
#using pdfplumber to locate annex page from ICSG report programmatically in order to extract production and refinery data
import pdfplumber

pdf_path = "../data/raw/icsg_factbook_2025_raw_annex.pdf"

with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages):
        text = page.extract_text() or ""

        if "WORLD COPPER PRODUCTION AND REFINED COPPER USAGE" in text:
            print("Found annex on page:", page_number)
            print(text[:3000])

In [54]:
#inspecting raw extracted table
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[page_number]

    table = page.extract_table(
        table_settings={
            "vertical_strategy": "text",
            "horizontal_strategy": "text"
        }
    )

for row in table:
    print(row)

TypeError: 'NoneType' object is not iterable

In [55]:
#pdfplumber likely did not detect targetted table, undertaking a diagnostic check
if table is None:
    print("No table detected on this page.")
else:
    for row in table:
        print(row)

No table detected on this page.


In [57]:
#inspecting the page as text
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[page_number]

    text = page.extract_text(layout=True)

print(text)

In [58]:
#page_number is likely pointing to the last PDF page. Assuming the cause is that page_number was used as the loop variable whilst searching, i.e., no page match or after finding the match, the loop continued.
#attempting to use a separate variable and stop once the annex is found
import pdfplumber

annex_page_number = None

with pdfplumber.open(pdf_path) as pdf:
    for page_index, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        normalized_text = " ".join(text.split()).lower()

        if (
            "world copper production and refined copper usage"
            in normalized_text
        ):
            annex_page_number = page_index
            break

if annex_page_number is None:
    raise ValueError("Annex page was not found")

print("Python page index:", annex_page_number)
print("PDF page number:", annex_page_number + 1)

Python page index: 1
PDF page number: 2


In [59]:
with pdfplumber.open(pdf_path) as pdf:
    annex_page = pdf.pages[annex_page_number]
    print(annex_page.extract_text(layout=True))

                                                                                                                    
                                                                                                                    
                                                                                                                    
                                             The World Copper Factbook 2025                                         
                                                                                                                    
                                                                                                                    
          Table of Contents                                  Chapter 5: Copper Trade ........................................................................................ 28
                                                              International Trade Flow of Copper Ores and Concentrates, 2

In [60]:
#that didn't work lol, let me use more distinctive text from the annex
annex_candidates = []

with pdfplumber.open(pdf_path) as pdf:
    for page_index, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        normalized_text = " ".join(text.split()).lower()

        if (
            "world copper production and refined copper usage"
            in normalized_text
            and "thousand metric tonnes copper"
            in normalized_text
            and "mine production" in normalized_text
            and "refined usage" in normalized_text
        ):
            annex_candidates.append(page_index)

print("Candidate pages:", annex_candidates)
print(
    "PDF page numbers:",
    [page_index + 1 for page_index in annex_candidates]
)

Candidate pages: []
PDF page numbers: []


In [61]:
#no candidate pages, inspecting all pages containing the broad phrase
with pdfplumber.open(pdf_path) as pdf:
    for page_index, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        normalized_text = " ".join(text.split()).lower()

        if "world copper production" in normalized_text:
            print(
                "Python index:",
                page_index,
                "PDF page:",
                page_index + 1
            )

Python index: 1 PDF page: 2


In [62]:
#the broad search finds only the table of contents; the stricter search finds nothing. Thus, the annex page must be either on a different page than expected or its table content is not exposed to pdfplumber as ordinary text.
#first, let's inspect the last several pages directly. The annex should be near the end of the 68 pg PDF.
import pdfplumber

with pdfplumber.open(pdf_path) as pdf:
    print("Total pages:", len(pdf.pages))

    for page_index in range(max(0, len(pdf.pages) - 10), len(pdf.pages)):
        text = pdf.pages[page_index].extract_text() or ""

        print("\n--- Python index:", page_index,
              "| PDF page:", page_index + 1, "---")
        print(text[:500])

Total pages: 68

--- Python index: 58 | PDF page: 59 ---
The World Copper Factbook 2025
C R R D “metallurgical” indicator are the metal industry, metal traders, and
OPPER ECYCLING ATE EFINITIONS
resource policymakers. However, given structural and process
variables, it may have limited use as a policy tool.
•• The Overall Recycling Efficiency Rate indicates the efficiency with
which end-of-life (EoL) scrap, new scrap, and other metal-bearing
residues are collected and recycled by a network of collectors,
processors, and metal recyclers. The key target

--- Python index: 59 | PDF page: 60 ---
The World Copper Factbook 2025
ICSG G C S U R I R , 2005-2024
LOBAL OPPER CRAP SAGE AND ECYCLING NPUT ATE
Million metric tonnes of copper
Source: ICSG
Direct-melt copper scrap figures for 2024 are preliminary estimates, as is the Recycling Input Rate (RIR).
In ternatio nal Cop per Stu dy Group 55
(
3
3
2
2
2
1
1
M
6
2
8
4
0
6
2
8
4
0
t C
5002
u )
6002 7002
D
( S e
8002
ir e
c
c
o
t
n
9002
M
d a
e
r

In [63]:
#found it on python index: 64, and pdf page: 65. However, data is poorly extracted. Distorted headings are likely a layout extraction issue, as opposed to a data issue. Parsing now:
with pdfplumber.open(pdf_path) as pdf:
    annex_page = pdf.pages[64]
    annex_text = annex_page.extract_text(layout=False)

print(annex_text)

The World Copper Factbook 2025
ANNEX
W C P R C U , 1960-2024
ORLD OPPER RODUCTION AND EFINED OPPER SAGE
Thousand Metric Tonnes Copper
Source: ICSG
Mine Refined Refined Mine R efined Refined Mine Refined Refined
Production Production Usage Production Production Usage Production Production Usage
1960 3,924 4,998 4,738 1982 7,745 9,319 9,090 2004 14,594 15,918 16,743
1961 4,081 5,127 5,050 1983 7,824 9,541 9,510 2005 14,927 16,572 16,552
1962 4,216 5,296 5,048 1984 8,135 9,440 9,930 2006 14,983 17,288 16,917
1963 4,286 5,400 5,500 1985 8,314 9,616 9,798 2007 15,508 17,895 18,026
1964 4,443 5,739 5,995 1986 8,295 9,920 10,112 2008 15,532 18,191 17,877
1965 4,769 6,059 6,193 1987 8,620 10,148 10,293 2009 15,941 18,234 17,870
1966 4,987 6,324 6,445 1988 8,773 10,512 10,668 2010 15,987 18,965 19,136
1967 4,743 6,004 6,195 1989 9,086 10,908 11,081 2011 15,960 19,585 19,709
1968 5,010 6,653 6,523 1990 9,227 10,805 10,886 2012 16,678 20,169 20,479
1969 5,682 7,212 7,137 1991 9,373 10,686 10,563 

In [65]:
#numerical rows look properly aligned. Proceeding with extracting the data rows.
import re
import pandas as pd

records = []

for line in annex_text.splitlines():
    tokens = line.split()

    #data rows begin with a four-digit year
    if not tokens or not re.match(r"^\d{4}", tokens[0]):
        continue

    #each block contains year + three values
    for position in range(0, len(tokens), 4):
        block = tokens[position:position + 4]

        if len(block) !=4:
            continue

        year_token, mine, refined, usage = block

        year = int(re.sub(r"\D", "", year_token))

        records.append({
            "year": year,
            "copper_mine_production_kt": mine.replace(",", ""),
            "refined_copper_production_kt": refined.replace(",", ""),
            "refined_copper_usage_kt": usage.replace(",", ""),
            "preliminary": "/p" in year_token.lower()
        })

In [66]:
#converting the values to numeric types
annual = pd.DataFrame(records)

numeric_columns = [
    "copper_mine_production_kt",
    "refined_copper_production_kt",
    "refined_copper_usage_kt"
]

for column in numeric_columns:
    annual[column] = pd.to_numeric(
        annual[column],
        errors="coerce"
    )

annual["date"] = pd.to_datetime(
    annual["year"].astype(str) + "-12-31"
)

annual = annual[
    [
        "date",
        "year",
        "copper_mine_production_kt",
        "refined_copper_production_kt",
        "refined_copper_usage_kt",
        "preliminary"
    ]
].sort_values("year")

In [67]:
#validating
print(annual.head())
print(annual.tail())
print(annual.shape)
print(annual["year"].is_unique)
print(annual["year"].min(), annual["year"].max())

         date  year  copper_mine_production_kt  refined_copper_production_kt  \
0  1960-12-31  1960                       3924                          4998   
3  1961-12-31  1961                       4081                          5127   
6  1962-12-31  1962                       4216                          5296   
9  1963-12-31  1963                       4286                          5400   
12 1964-12-31  1964                       4443                          5739   

    refined_copper_usage_kt  preliminary  
0                      4738        False  
3                      5050        False  
6                      5048        False  
9                      5500        False  
12                     5995        False  
         date  year  copper_mine_production_kt  refined_copper_production_kt  \
50 2020-12-31  2020                      20740                         24621   
53 2021-12-31  2021                      21223                         24900   
56 2022-12-31  2022  

In [68]:
#index retains the original extraction order, reseting it
annual = (
    annual
    .sort_values("year")
    .reset_index(drop=True)
)

In [69]:
#inspecting key recent years
recent = annual[annual["year"] >= 2019].copy()

recent

,date,year,copper_mine_production_kt,refined_copper_production_kt,refined_copper_usage_kt,preliminary
59,2019-12-31,2019,20657,24127,24351,False
60,2020-12-31,2020,20740,24621,24953,False
61,2021-12-31,2021,21223,24900,25259,False
62,2022-12-31,2022,21911,25272,25857,False
63,2023-12-31,2023,22368,26502,26604,False
64,2024-12-31,2024,22990,27486,27353,True


In [70]:
#calculating refined copper balance
recent["refined_copper_balance_kt"] = (
    recent["refined_copper_production_kt"]
    - recent["refined_copper_usage_kt"]
)

recent[
    [
        "year",
        "refined_copper_production_kt",
        "refined_copper_usage_kt",
        "refined_copper_balance_kt",
        "preliminary"
    ]
]

,year,refined_copper_production_kt,refined_copper_usage_kt,refined_copper_balance_kt,preliminary
59,2019,24127,24351,-224,False
60,2020,24621,24953,-332,False
61,2021,24900,25259,-359,False
62,2022,25272,25857,-585,False
63,2023,26502,26604,-102,False
64,2024,27486,27353,133,True


In [71]:
annual.to_csv(
    "../data/processed/"
    "icsg_world_copper_production_usage_annual_clean.csv",
    index=False
)

In [73]:
#validating against raw statisa dataset, treating it as an independent validation source
#loading the two datasets
import pandas as pd
import numpy as np

icsg = pd.read_csv(
    "../data/processed/"
    "icsg_world_copper_production_usage_annual_clean.csv"
)

statista = pd.read_csv(
    "../data/raw/"
    "statista_global_refinery_copper_production_kt_annual_raw.csv"
)

In [74]:
#standardizing statista file
statista.columns = [
    "year",
    "refined_copper_production_kt_statista"
]

statista["year"] = (
    statista["year"]
    .astype(str)
    .str.replace("*", "", regex=False)
    .astype(int)
)

statista["refined_copper_production_kt_statista"] = (
    pd.to_numeric(
        statista["refined_copper_production_kt_statista"],
        errors="coerce"
    )
)

In [75]:
#merging relevant ICSG column
comparison = icsg[
    [
        "year",
        "refined_copper_production_kt"
    ]
].merge(
    statista,
    on="year",
    how="outer",
    indicator=True
)

In [76]:
#calculating differences
comparison["difference_kt"] = (
    comparison["refined_copper_production_kt"]
    - comparison["refined_copper_production_kt_statista"]
)

comparison["percentage_difference"] = (
    comparison["difference_kt"]
    / comparison["refined_copper_production_kt_statista"]
    * 100
)

In [77]:
comparison

,year,refined_copper_production_kt,refined_copper_production_kt_statista,_merge,difference_kt,percentage_difference
0,1960,4998,NaN,left_only,NaN,NaN
1,1961,5127,NaN,left_only,NaN,NaN
2,1962,5296,NaN,left_only,NaN,NaN
3,1963,5400,NaN,left_only,NaN,NaN
4,1964,5739,NaN,left_only,NaN,NaN
...,...,...,...,...,...,...
60,2020,24621,24621.0,both,0.0,0.0
61,2021,24900,24900.0,both,0.0,0.0
62,2022,25272,25272.0,both,0.0,0.0
63,2023,26502,26502.0,both,0.0,0.0


In [78]:
#no differences upon manual inspection
#checking whether all overlapping values match exactly
overlap = comparison[
    comparison["_merge"] == "both"
]

print("Maximum absolute difference:",
      overlap["difference_kt"].abs().max())

print("Number of differences:",
      (overlap["difference_kt"] != 0).sum())

Maximum absolute difference: 0.0
Number of differences: 0


In [81]:
#two files match exactly per output above.
#formally verifying it:
assert overlap["difference_kt"].eq(0).all()

In [82]:
#checking non overlapping years:
comparison[
    comparison["_merge"] != "both"
][["year", "_merge"]]

,year,_merge
0,1960,left_only
1,1961,left_only
2,1962,left_only
3,1963,left_only
4,1964,left_only
5,1965,left_only
6,1966,left_only
7,1967,left_only
8,1968,left_only
9,1969,left_only


In [83]:
#saving the comparison as a validation artifact
comparison.to_csv(
    "../data/processed/"
    "icsg_vs_statista_production_validation.csv",
    index=False
)

In [88]:
#running final checks
assert annual["year"].between(1960, 2024).all()
assert annual["year"].is_unique
assert annual[numeric_columns].notna().all().all()
assert annual["year"].min() == 1960
assert annual["year"].max() == 2024

#calculating derived refined balance
annual["refined_copper_balance_kt"] = (
    annual["refined_copper_production_kt"]
    - annual["refined_copper_usage_kt"]
)

annual_analysis = annual.copy()

annual_analysis["refined_copper_balance_kt"] = (
    annual_analysis["refined_copper_production_kt"]
    - annual_analysis["refined_copper_usage_kt"]
)

#exporting derived analytical output as a .csv file
annual_analysis.to_csv(
    "../data/processed/"
    "copper_fundamentals_annual_analysis.csv",
    index=False
)

In [89]:
#saving a final quality-control summary:
validation_summary = {
    "observations": len(annual),
    "start_year": annual["year"].min(),
    "end_year": annual["year"].max(),
    "duplicate_years": annual["year"].duplicated().sum(),
    "missing_values": annual.isna().sum().sum(),
    "statista_production_discrepancies": (
        comparison["difference_kt"] != 0
    ).sum()
}

pd.Series(validation_summary).to_csv(
    "../data/processed/"
    "copper_fundamentals_validation_summary.csv"
)

In [92]:
# Used Copper Production: World Refined Copper Production Mountain graph, p. 22, from ICSG's World Copper Factbook 2025, in conjunction with WebPlotDigitizer, to draw two boundaries. First one drawn at values of Refinery Primary; second at values of Refinery Secondary. Thus, I will take the difference between the secondary and primary, in order to approximate values for secondary/recycled copper production; then validate using publicly available data.
# Importing corresponding .csv files for both primary and secondary values. The files do not contain column headers; therefore providing column names myself.
import pandas as pd
import numpy as np

primary = pd.read_csv(
    "../data/raw/primary_cumulative_boundary_mt.csv",
    header=None,
    names=["year_raw", "primary_boundary_mt"]
)

secondary_top = pd.read_csv(
    "../data/raw/primary_plus_secondary_boundary_mt.csv",
    header=None,
    names=[
        "year_raw",
        "primary_plus_secondary_boundary_mt"
    ]
)

In [93]:
# Inspecting assigned column names.
print(primary.head())
print(secondary_top.head())
print(primary.shape)
print(secondary_top.shape)

      year_raw  primary_boundary_mt
0  1960.091822             3.733333
1  1961.010043             3.845161
2  1962.020086             3.938351
3  1963.011765             4.087455
4  1964.003443             4.348387
      year_raw  primary_plus_secondary_boundary_mt
0  1960.036729                            4.422939
1  1961.028407                            4.572043
2  1962.020086                            4.646595
3  1963.030129                            4.944803
4  1964.040172                            5.205735
(65, 2)
(65, 2)


In [94]:
# Output looks about right.
# Converting fields.
import numpy as np

for df in [primary, secondary_top]:
    df["year_raw"] = pd.to_numeric(
        df["year_raw"],
        errors="coerce"
    )

primary["primary_boundary_mt"] = pd.to_numeric(
    primary["primary_boundary_mt"],
    errors="coerce"
)

secondary_top[
    "primary_plus_secondary_boundary_mt"
] = pd.to_numeric(
    secondary_top[
        "primary_plus_secondary_boundary_mt"
    ],
    errors="coerce"
)

In [95]:
# Creating integer years.
primary["year"] = np.rint(
    primary["year_raw"]
).astype(int)

secondary_top["year"] = np.rint(
    secondary_top["year_raw"]
).astype(int)

In [96]:
# Retaining the fields needed for merging
primary_for_merge = primary[
    ["year", "primary_boundary_mt"]
].copy()

secondary_for_merge = secondary_top[
    [
        "year",
        "primary_plus_secondary_boundary_mt"
    ]
].copy()

In [97]:
# Merging the two extracted boundaries.

digitized = primary_for_merge.merge(
    secondary_for_merge,
    on="year",
    how="outer",
    indicator=True,
    validate="one_to_one"
)

In [98]:
# Ensuring that each year appears in both files.

print(digitized["_merge"].value_counts())

if not digitized["_merge"].eq("both").all():
    print(
        digitized[
            digitized["_merge"] != "both"
        ]
    )
    raise ValueError(
        "Some years do not appear in both extracted datasets."
    )

_merge
both          65
left_only      0
right_only     0
Name: count, dtype: int64


In [99]:
# Approximating secondary refined production

digitized["secondary_refined_production_mt"] = (
    digitized[
        "primary_plus_secondary_boundary_mt"
    ]
    - digitized["primary_boundary_mt"]
)

In [100]:
# Converting from million tonnes to thousand tonnes.

digitized["secondary_refined_production_kt"] = (
    digitized["secondary_refined_production_mt"] * 1000
)

In [101]:
# Adding standardized dates and sorting.

digitized["date"] = pd.to_datetime(
    digitized["year"].astype(str) + "-12-31"
)

digitized = (
    digitized
    .sort_values("year")
    .reset_index(drop=True)
)

In [102]:
# Quick quality checks.

assert len(digitized) == 65
assert digitized["year"].min() == 1960
assert digitized["year"].max() == 2024
assert digitized["year"].is_unique
assert digitized[
    "secondary_refined_production_mt"
].ge(0).all()

In [103]:
# Inspecting results.

digitized[
    [
        "year",
        "primary_boundary_mt",
        "primary_plus_secondary_boundary_mt",
        "secondary_refined_production_mt",
        "secondary_refined_production_kt"
    ]
].head()

,year,primary_boundary_mt,primary_plus_secondary_boundary_mt,secondary_refined_production_mt,secondary_refined_production_kt
0,1960,3.733333,4.422939,0.689606,689.605735
1,1961,3.845161,4.572043,0.726882,726.881720
2,1962,3.938351,4.646595,0.708244,708.243728
3,1963,4.087455,4.944803,0.857348,857.347670
4,1964,4.348387,5.205735,0.857348,857.347670


In [104]:
# Inspecting recent years.

digitized[
    digitized["year"] >= 2020
]

,year,primary_boundary_mt,primary_plus_secondary_boundary_mt,_merge,secondary_refined_production_mt,secondary_refined_production_kt,date
60,2020,16.649462,20.488889,both,3.839427,3839.426523,2020-12-31
61,2021,16.761290,20.898925,both,4.137634,4137.634409,2021-12-31
62,2022,16.873118,20.973477,both,4.100358,4100.358423,2022-12-31
63,2023,17.469534,21.961290,both,4.491756,4491.756272,2023-12-31
64,2024,18.065950,22.762724,both,4.696774,4696.774194,2024-12-31


In [105]:
# Outputs seem structurally correct and economically plausible. 2024 figure is especially reassuring, relative to total refined production, i.e. secondary share is approximately 17.1%, which matches the figure quoted in ICSG's Factbook.
# Running additional checks.

assert digitized["_merge"].eq("both").all()
assert digitized["secondary_refined_production_mt"].ge(0).all()
assert digitized["year"].is_unique
assert digitized["year"].min() == 1960
assert digitized["year"].max() == 2024

In [106]:
# Since the values were digitized from a chart, I won't present them with excessive precision. Rather, I'll preserve the raw extracted values, but create a rounded reporting figure in application.
digitized[
    "secondary_refined_production_mt_rounded"
] = digitized[
    "secondary_refined_production_mt"
].round(2)

digitized[
    "secondary_refined_production_kt_rounded"
] = (
    digitized[
        "secondary_refined_production_kt"
    ].round(-1)
)

In [107]:
# Removing merge-status column from final output.
secondary_clean = digitized[
    [
        "date",
        "year",
        "secondary_refined_production_mt",
        "secondary_refined_production_kt",
        "secondary_refined_production_mt_rounded",
        "secondary_refined_production_kt_rounded"
    ]
].copy()

In [108]:
# Saving as a digitized derived series.
secondary_clean.to_csv(
    "../data/processed/"
    "secondary_refined_production_annual_digitized.csv",
    index=False
)

In [1]:
#Extracting copper stock data from copper council report PDF.
import pdfplumber
import re
import pandas as pd

pdf_path = "../data/raw/copper_council_stock_tables_raw.pdf"

target_page = None

with pdfplumber.open(pdf_path) as pdf:
    for page_index, page in enumerate(pdf.pages):
        text = page.extract_text() or ""

        if "Stocks at Year/Month end" in text:
            target_page = page_index
            print("Found table on PDF page:", page_index + 1)
            break

if target_page is None:
    raise ValueError("Stocks table was not found")

Found table on PDF page: 5


In [2]:
#Inspecting target page.
with pdfplumber.open(pdf_path) as pdf:
    table_text = pdf.pages[target_page].extract_text(
        layout=False
    )

print(table_text)

LME, Comex, and Shanghai
Tonnes
Stocks at Year/Month end
LME Comex Shanghai Total
1999 790,225 83,097 63,213 936,535
2000 357,225 58,669 112,117 528,011
2001 799,225 243,806 94,487 1,137,518
2002 856,400 362,276 75,087 1,293,763
2003 430,525 254,863 120,631 806,019
2004 48,875 48,203 31,685 128,763
2005 96,175 6,182 57,844 160,201
2006 190,575 30,915 31,300 252,790
2007 198,925 13,816 25,597 238,338
2008 340,550 31,311 15,326 387,187
2009 502,400 90,230 96,362 688,992
2010 377,675 58,923 131,891 568,489
2011 371,575 79,817 93,219 544,611
2012 320,500 64,149 204,773 589,422
2013 365,700 14,915 125,849 506,464
2014 177,025 23,890 111,915 312,830
2015 236,225 63,233 182,835 482,293
2016 311,825 80,651 146,598 539,074
2017 200,650 191,572 150,489 542,711
2018 132,175 99,868 118,686 350,729
2019 144,675 34,065 123,647 302,387
2020 105,800 70,397 120,112 296,309
2021 88,725 63,201 45,162 197,088
2022 88,925 31,834 99,265 220,024
Jan-19 149,950 77,991 119,727 347,668
Feb-19 126,100 52,280 217

In [3]:
#Extracting annual rows.
pattern = re.compile(
    r"^\s*(\d{4})\s+"
    r"([\d,]+)\s+"
    r"([\d,]+)\s+"
    r"([\d,]+)\s+"
    r"([\d,]+)\s*$"
)

records = []

for line in table_text.splitlines():
    match = pattern.match(line)

    if match:
        year, lme, comex, shanghai, total = match.groups()

        year = int(year)

        if 2000 <= year <= 2022:
            records.append({
                "year": year,
                "lme_copper_inventory_tonnes": int(
                    lme.replace(",", "")
                ),
                "comex_copper_inventory_tonnes": int(
                    comex.replace(",", "")
                ),
                "shfe_copper_inventory_tonnes": int(
                    shanghai.replace(",", "")
                ),
                "reported_total_inventory_tonnes": int(
                    total.replace(",", "")
                )
            })

In [4]:
# Creating Data Frame
inventory_annual = (
    pd.DataFrame(records)
    .sort_values("year")
    .reset_index(drop=True)
)

In [5]:
# Validating Data Frame
assert len(inventory_annual) == 23
assert inventory_annual["year"].is_unique
assert inventory_annual["year"].min() == 2000
assert inventory_annual["year"].max() == 2022

In [7]:
# Verifying reported totals. Zero Differences should be obtained.
inventory_annual[
    "calculated_total_inventory_tonnes"
] = (
    inventory_annual["lme_copper_inventory_tonnes"]
    + inventory_annual["comex_copper_inventory_tonnes"]
    + inventory_annual["shfe_copper_inventory_tonnes"]
)

inventory_annual["total_difference_tonnes"] = (
    inventory_annual["calculated_total_inventory_tonnes"]
    - inventory_annual["reported_total_inventory_tonnes"]
)

inventory_annual[
    "total_difference_tonnes"
].value_counts()

total_difference_tonnes
0    23
Name: count, dtype: int64

In [8]:
# Saving extracted raw annual series.
inventory_annual.to_csv(
    "../data/processed/"
    "copper_exchange_inventories_annual_2000_2022_clean.csv",
    index=False
)

In [2]:
# Safely inspecting .xls dataset from Cochilco, Chilean Copper Commission.
from pathlib import Path
import hashlib
import pandas as pd
import xlrd

xls_path = Path(r"../data/raw/cochilco_copper_inventories_exchanges.xls")

# Recording a reproducibility hash.
sha256 = hashlib.sha256(xls_path.read_bytes()).hexdigest()
print("SHA-256", sha256)

# Inspecting workbook strcuture.
workbook = pd.ExcelFile(xls_path, engine="xlrd")
print(workbook.sheet_names)

# Reading without inspecting headers.
raw = pd.read_excel(
    xls_path,
    sheet_name=0,
    header=None,
    engine="xlrd"
)

print(raw.shape)
display(raw.head(25))

SHA-256 84e848276b4bbabd99218f44140ebe0ea990e425e4c4abd4858eae5dcaab0cfc


XLRDError: Unsupported format, or corrupt file: Expected BOF record; found b'<!-- Goo'

In [3]:
# Apparently the file should be HTML?
# Inspecting
print(xls_path.read_bytes()[:500].decode("utf-8", errors="replace"))

<!-- Google tag (gtag.js) -->
<script async src="https://www.googletagmanager.com/gtag/js?id=G-CJEMCG1KTJ"></script>
<script>
  window.dataLayer = window.dataLayer || [];
  function gtag(){dataLayer.push(arguments);}
  gtag('js', new Date());

  gtag('config', 'G-CJEMCG1KTJ');
</script>



<!-- Google tag (gtag.js) -->
<script async src="https://www.googletagmanager.com/gtag/js?id=G-CJEMCG1KTJ"></script>
<script>
  window.dataLayer = window.dataLayer || [];
  function gtag(){data


In [4]:
# Website analytics? Checking whether the page contains an embedded table.
from io import StringIO
import pandas as pd
html_text = xls_path.read_text(
    encoding="utf-8",
    errors="replace"
)

tables = pd.read_html(StringIO(html_text))

print("Tables found:", len(tables))

for i, table in enumerate(tables):
    print(i, table.shape)
    display(table.head())

Tables found: 4
0 (1, 1)


,0
0,TABLA / TABLE 4.1


1 (3, 1)


,0
0,INVENTARIOS DE COBRE EN BOLSAS - ANUAL Y MENSUAL
1,Exchange Inventories (Yearly and Monthly)
2,(TM Fin de Período / MT at Close)


2 (16, 9)


,0,1,2,3,4,5,6,7,8
0,INVENTARIOS A FIN DE PERIODO / EXCHANGE INVENT...,BML / LME,BML / LME,COMEX,COMEX,SHFE,SHFE,NaN,NaN
1,INVENTARIOS A FIN DE PERIODO / EXCHANGE INVENT...,LONDON METAL EXCHANGE,LONDON METAL EXCHANGE,NEW YORK MERCANTILE EXCHANGE,NEW YORK MERCANTILE EXCHANGE,SHANGHAI FUTURES EXCHANGE,SHANGHAI FUTURES EXCHANGE,NaN,NaN
2,INVENTARIOS A FIN DE PERIODO / EXCHANGE INVENT...,TOTAL,Variación / Change,TOTAL,Variación / Change,TOTAL,Variación / Change,TOTAL,Variación / Change
3,2022 2023 2024 2025,88.925 167.300 271.400 147.425,-25 78.375 104.100 -123.975,31.834 17.064 84.514 451.833,-31.367 -14.770 67.450 367.319,69.268 30.905 74.172 111.703,31.086 -38.363 43.267 37.531,190.027 215.269 430.086 710.961,-306 25.242 214.817 280.875
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


3 (1, 1)


,0
0,Fuente / Source: Elaborado por la Comisión Chi...


In [5]:
# Table 2 is the relevant exchange-inventory table I was looking for. Duplicated labels likely refer to the same exchange, BML = Bolsa Metales de Londres, i.e., London Metal Exchange.
# Since the webpage uses merged cells, annual and monthly observations seem to have been compressed into multiline cells.
# Inspecting the entire table.
exchange_table = tables[2].copy()

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

display(exchange_table)

,0,1,2,3,4,5,6,7,8
0,INVENTARIOS A FIN DE PERIODO / EXCHANGE INVENTORIES AT CLOSE,BML / LME,BML / LME,COMEX,COMEX,SHFE,SHFE,NaN,NaN
1,INVENTARIOS A FIN DE PERIODO / EXCHANGE INVENTORIES AT CLOSE,LONDON METAL EXCHANGE,LONDON METAL EXCHANGE,NEW YORK MERCANTILE EXCHANGE,NEW YORK MERCANTILE EXCHANGE,SHANGHAI FUTURES EXCHANGE,SHANGHAI FUTURES EXCHANGE,NaN,NaN
2,INVENTARIOS A FIN DE PERIODO / EXCHANGE INVENTORIES AT CLOSE,TOTAL,Variación / Change,TOTAL,Variación / Change,TOTAL,Variación / Change,TOTAL,Variación / Change
3,2022 2023 2024 2025,88.925 167.300 271.400 147.425,-25 78.375 104.100 -123.975,31.834 17.064 84.514 451.833,-31.367 -14.770 67.450 367.319,69.268 30.905 74.172 111.703,31.086 -38.363 43.267 37.531,190.027 215.269 430.086 710.961,-306 25.242 214.817 280.875
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ENE/JAN 2023 FEB MAR ABR/APR MAY JUN JUL AGO/AUG SEP OCT NOV DIC/DEC,75.225 64.100 64.725 64.550 99.150 72.975 68.350 102.900 167.825 176.475 175.250 167.300,-13.700 -11.125 625 -175 34.600 -26.175 -4.625 34.550 64.925 8.650 -1.225 -7.950,25.894 15.487 15.008 25.081 25.131 30.780 39.426 29.396 22.987 19.886 17.831 17.064,-5.940 -10.407 -479 10.073 50 5.649 8.646 -10.030 -6.409 -3.101 -2.055 -767,139.967 252.455 156.576 137.095 86.648 68.313 61.290 46.591 38.996 36.408 26.149 30.905,70.699 112.488 -95.879 -19.481 -50.447 -18.335 -7.023 -14.699 -7.595 -2.588 -10.259 4.756,241.086 332.042 236.309 226.726 210.929 172.068 169.066 178.887 229.808 232.769 219.230 215.269,51.059 90.956 -95.733 -9.583 -15.797 -38.861 -3.002 9.821 50.921 2.961 -13.539 -3.961
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ENE/JAN 2024 FEB MAR ABR/APR MAY JUN JUL AGO/AUG SEP OCT NOV DIC/DEC,146.475 122.675 112.475 116.126 116.475 180.125 239.275 320.925 300.600 271.375 271.000 271.400,-20.825 -23.800 -10.200 3.651 349 63.650 59.150 81.650 -20.325 -29.225 -375 400,21.993 26.336 27.269 22.135 15.066 8.206 13.354 36.394 57.647 80.485 82.278 84.514,4.929 4.343 933 -5.134 -7.069 -6.860 5.148 23.040 21.253 22.838 1.793 2.236,50.532 214.487 290.228 287.498 321.695 319.521 301.203 241.745 141.625 153.221 108.775 74.172,19.627 163.955 75.741 -2.730 34.197 -2.174 -18.318 -59.458 -100.120 11.596 -44.446 -34.603,219.000 363.498 429.972 425.759 453.236 507.852 553.832 599.064 499.872 505.081 462.053 430.086,3.731 144.498 66.474 -4.213 27.477 54.616 45.980 45.232 -99.192 5.209 -43.028 -31.967
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ENE/JAN 2025 FEB MAR ABR/APR MAY JUN JUL AGO/AUG SEP OCT NOV DIC/DEC,256.225 262.075 211.375 200.150 149.875 90.625 138.200 158.900 143.400 134.625 159.425 147.425,-15.175 5.850 -50.700 -11.225 -50.275 -59.250 47.575 20.700 -15.500 -8.775 24.800 -12.000,89.119 84.805 87.249 130.656 163.864 191.606 233.977 252.055 294.007 322.649 379.863 451.833,4.605 -4.314 2.444 43.407 33.208 27.742 42.371 18.078 41.952 28.642 57.214 71.970,98.049 268.337 235.296 89.307 105.791 81.550 72.543 79.748 95.034 116.140 97.930 111.703,23.877 170.288 -33.041 -145.989 16.484 -24.241 -9.007 7.205 15.286 21.106 -18.210 13.773,443.393 615.217 533.920 420.113 419.530 363.781 444.720 490.703 532.441 573.414 637.218 710.961,13.307 171.824 -81.297 -113.807 -583 -55.749 80.939 45.983 41.738 40.973 63.804 73.743


In [6]:
# Undertaking a clearer structural inspection.
for row_number, row in exchange_table.iterrows():
    print(f"\nROW {row_number}")

    for column_number, value in row.items():
        if pd.notna(value):
            print(f"COLUMN {column_number}: {repr(value)}")


ROW 0
COLUMN 0: 'INVENTARIOS A FIN DE PERIODO / EXCHANGE INVENTORIES AT CLOSE'
COLUMN 1: 'BML / LME'
COLUMN 2: 'BML / LME'
COLUMN 3: 'COMEX'
COLUMN 4: 'COMEX'
COLUMN 5: 'SHFE'
COLUMN 6: 'SHFE'

ROW 1
COLUMN 0: 'INVENTARIOS A FIN DE PERIODO / EXCHANGE INVENTORIES AT CLOSE'
COLUMN 1: 'LONDON METAL EXCHANGE'
COLUMN 2: 'LONDON METAL EXCHANGE'
COLUMN 3: 'NEW YORK MERCANTILE EXCHANGE'
COLUMN 4: 'NEW YORK MERCANTILE EXCHANGE'
COLUMN 5: 'SHANGHAI FUTURES EXCHANGE'
COLUMN 6: 'SHANGHAI FUTURES EXCHANGE'

ROW 2
COLUMN 0: 'INVENTARIOS A FIN DE PERIODO / EXCHANGE INVENTORIES AT CLOSE'
COLUMN 1: 'TOTAL'
COLUMN 2: 'Variación / Change'
COLUMN 3: 'TOTAL'
COLUMN 4: 'Variación / Change'
COLUMN 5: 'TOTAL'
COLUMN 6: 'Variación / Change'
COLUMN 7: 'TOTAL'
COLUMN 8: 'Variación / Change'

ROW 3
COLUMN 0: '2022  2023  2024  2025'
COLUMN 1: '88.925  167.300  271.400  147.425'
COLUMN 2: '-25  78.375  104.100  -123.975'
COLUMN 3: '31.834  17.064  84.514  451.833'
COLUMN 4: '-31.367  -14.770  67.450  367.319'
COL

In [7]:
# Inspection confirms that the table is structured coherently enough to nornalize the values.
# Using rows 5, 7, 9, and 11 for required monthly values, i.e., years 2023-2025 + 2026 YTD.
# Extracting LME TOTAL column and converting the multiline blocks into one observation per month:
import re
import pandas as pd

def split_tokens(value):
    if pd.isna(value):
        return []
    return str(value).replace("\n", " ").split()

monthly_records = []

for row_index, row in exchange_table.iterrows():
    label_text = " ".join(split_tokens(row.iloc[0]))
    lme_values = split_tokens(row.iloc[1])

    year_match = re.search(r"\b(20\d{2})\b", label_text)
    has_month_label = re.search(
        r"JAN|FEB|MAR|APR|MAY|JUN|JUL|AGO|AUG|SEP|OCT|NOV|DIC|DEC",
        label_text,
        flags=re.IGNORECASE
    )

    # Monthly rows contain a year, month labels, and at least four values.
    if year_match and has_month_label and 4 <= len(lme_values) <= 12:
        year = int(year_match.group(1))

        for month_number, value in enumerate(lme_values, start=1):
            # Periods are thousands separators.
            # "75.225" means 75,225 tonnes.
            inventory_tonnes = int(value.replace(".", ""))

            date = (
                pd.Timestamp(year=year, month=month_number, day=1)
                + pd.offsets.MonthEnd(0)
            )

            monthly_records.append({
                "date": date,
                "lme_copper_inventory_tonnes": inventory_tonnes,
                "source": "Cochilco Table 4.1; underlying Reuters data",
                "source_row": row_index
            })

lme_monthly = (
    pd.DataFrame(monthly_records)
    .sort_values("date")
    .drop_duplicates("date")
    .reset_index(drop=True)
)

display(lme_monthly)
print("Observations:", len(lme_monthly))
print("Date range:", lme_monthly["date"].min(), "to", lme_monthly["date"].max())

,date,lme_copper_inventory_tonnes,source,source_row
0,2023-01-31,75225,Cochilco Table 4.1; underlying Reuters data,5
1,2023-02-28,64100,Cochilco Table 4.1; underlying Reuters data,5
2,2023-03-31,64725,Cochilco Table 4.1; underlying Reuters data,5
3,2023-04-30,64550,Cochilco Table 4.1; underlying Reuters data,5
4,2023-05-31,99150,Cochilco Table 4.1; underlying Reuters data,5
5,2023-06-30,72975,Cochilco Table 4.1; underlying Reuters data,5
6,2023-07-31,68350,Cochilco Table 4.1; underlying Reuters data,5
7,2023-08-31,102900,Cochilco Table 4.1; underlying Reuters data,5
8,2023-09-30,167825,Cochilco Table 4.1; underlying Reuters data,5
9,2023-10-31,176475,Cochilco Table 4.1; underlying Reuters data,5


Observations: 40
Date range: 2023-01-31 00:00:00 to 2026-04-30 00:00:00


In [11]:
# Observations are as expected; 12 months each for 2023-2025, 4 months for 2026.
# Performing basic validation. Expected values are 167,300 for 2023, 271,400 for 2024, and 147,425 for 2025.
assert lme_monthly["date"].is_unique
assert lme_monthly["lme_copper_inventory_tonnes"].notna().all()

december = lme_monthly.loc[
    lme_monthly["date"].dt.month == 12
].copy()

december["year"] = december["date"].dt.year

december_values = (
    december
    .set_index("year")["lme_copper_inventory_tonnes"]
)

print(december_values)

year
2023    167300
2024    271400
2025    147425
Name: lme_copper_inventory_tonnes, dtype: int64


In [12]:
# Formalizing validation above.
expected_december = pd.Series({
    2023: 167300,
    2024: 271400,
    2025: 147425
}, name="lme_copper_inventory_tonnes")

pd.testing.assert_series_equal(
    december_values.sort_index(),
    expected_december.sort_index(),
    check_names=True
)

print("December validation passed.")

AssertionError: Series.index are different

Attribute "dtype" are different
[left]:  int32
[right]: int64

In [15]:
# Assertion likely fails because the index data types differ; actual index: int32; expected index: int64.
# Actual series has an index name ("year"), however the expected series has no index name. 
# Assigning the same index name and dtype explicitly to undertake a strict validation.
december_values = december_values.sort_index()
expected_december = expected_december.sort_index()

december_values.index = pd.Index(
    december_values.index,
    dtype="int64",
    name="year"
)

expected_december.index = pd.Index(
    expected_december.index,
    dtype="int64",
    name="year"
)

pd.testing.assert_series_equal(
    december_values,
    expected_december,
    check_dtype=True,
    check_names=True,
    check_exact=True
)

print("December validation passed.")

December validation passed.


In [16]:
# Performing final cleanup.
from pathlib import Path
import pandas as pd

lme_clean = lme_monthly[
    ["date", "lme_copper_inventory_tonnes", "source", "source_row"]
].copy()

lme_clean["date"] = pd.to_datetime(lme_clean["date"])
lme_clean["lme_copper_inventory_tonnes"] = (
    lme_clean["lme_copper_inventory_tonnes"]
    .astype("int64")
)

lme_clean = (
    lme_clean
    .sort_values("date")
    .drop_duplicates("date")
    .reset_index(drop=True)
)

# Final structural checks.
assert len(lme_clean) == 40
assert lme_clean["date"].is_unique
assert lme_clean["date"].notna().all()
assert lme_clean["lme_copper_inventory_tonnes"].notna().all()
assert (lme_clean["lme_copper_inventory_tonnes"] >= 0).all()

display(lme_clean)

,date,lme_copper_inventory_tonnes,source,source_row
0,2023-01-31,75225,Cochilco Table 4.1; underlying Reuters data,5
1,2023-02-28,64100,Cochilco Table 4.1; underlying Reuters data,5
2,2023-03-31,64725,Cochilco Table 4.1; underlying Reuters data,5
3,2023-04-30,64550,Cochilco Table 4.1; underlying Reuters data,5
4,2023-05-31,99150,Cochilco Table 4.1; underlying Reuters data,5
5,2023-06-30,72975,Cochilco Table 4.1; underlying Reuters data,5
6,2023-07-31,68350,Cochilco Table 4.1; underlying Reuters data,5
7,2023-08-31,102900,Cochilco Table 4.1; underlying Reuters data,5
8,2023-09-30,167825,Cochilco Table 4.1; underlying Reuters data,5
9,2023-10-31,176475,Cochilco Table 4.1; underlying Reuters data,5


In [17]:
# Adding a source ID, whilst retaining source_row for auditability.
lme_clean["source_id"] = "SRC-005"

lme_clean = lme_clean[
    [
        "date",
        "lme_copper_inventory_tonnes",
        "source_id",
        "source",
        "source_row"
    ]
]

In [18]:
# Saving processed dataset.
lme_clean.to_csv(
    r"..\data\processed\lme_copper_inventory_monthly_cochilco_clean.csv",
    index=False
)

In [21]:
# Concerning comparing and validating copper_exchange_inventories_annual_2000_2022_clean.csv and lme_copper_inventory_monthly_cochilco_clean.csv:
# The two files should not be compared as if they contain overlapping observations. The annual file ends in 2022, whilst the COCHILCO montly file begins in Jan 2023.
# Therefore, direct date-based validation will produce no overlapping dates.
# Firstly, let's confirm this formally. Expected output is "Overlapping dates: 0"
annual = pd.read_csv("../data/processed/copper_exchange_inventories_annual_2000_2022_clean.csv")
monthly = pd.read_csv("../data/processed/lme_copper_inventory_monthly_cochilco_clean.csv")

annual["date"] = pd.to_datetime(
    annual["year"].astype(str) + "-12-31"
)

annual_lme = annual[
    ["date", "lme_copper_inventory_tonnes"]
].copy()

monthly_lme = monthly[
    ["date", "lme_copper_inventory_tonnes"]
].copy()

overlap = annual_lme.merge(
    monthly_lme,
    on="date",
    how="inner",
    suffixes=("_annual", "_cochilco")
)

print("Overlapping dates:", len(overlap))

ValueError: You are trying to merge on datetime64[ns] and object columns for key 'date'. If you wish to proceed you should use pd.concat

In [22]:
# 'date' columns have different data types. annual date: datetime64[ns]; monthly date: object.
# Monthly dates, therefore, remained as strings as pd.read_csv() does not automatically convert date columns.
# Converting both explicitly, before merging.
annual_lme["date"] = pd.to_datetime(
    annual_lme["date"],
    errors="raise"
)

monthly_lme["date"] = pd.to_datetime(
    monthly_lme["date"],
    errors="raise"
)

In [23]:
# Standardizing both to month-end for consistency.
annual_lme["date"] = (
    annual_lme["date"]
    .dt.to_period("M")
    .dt.to_timestamp(how="end")
    .dt.normalize()
)

monthly_lme["date"] = (
    monthly_lme["date"]
    .dt.to_period("M")
    .dt.to_timestamp(how="end")
    .dt.normalize()
)

In [24]:
# Confirming that the types now match.
print(annual_lme["date"].dtype)
print(monthly_lme["date"].dtype)

datetime64[ns]
datetime64[ns]


In [25]:
# Running the overlap test. Expected output is "Overlapping dates: 0".
overlap = annual_lme.merge(
    monthly_lme,
    on="date",
    how="inner",
    suffixes=("_annual", "_cochilco"),
    validate="one_to_one"
)

print("Overlapping dates:", len(overlap))

Overlapping dates: 0


In [26]:
# Validating the annual series against COCHILCO's annual summary values.
# The COCHILCO table contains annual LME values for 2022-2025. Extracting from row 3:
annual_labels = split_tokens(exchange_table.iloc[3, 0])
annual_lme_values = split_tokens(exchange_table.iloc[3, 1])

cochilco_annual = pd.DataFrame({
    "year": [int(value) for value in annual_labels],
    "lme_copper_inventory_tonnes_cochilco": [
        int(value.replace(".", ""))
        for value in annual_lme_values
    ]
})

display(cochilco_annual)

,year,lme_copper_inventory_tonnes_cochilco
0,2022,88925
1,2023,167300
2,2024,271400
3,2025,147425


In [28]:
# Comparing the common year, 2022, with the historical annual dataset. Expected result should be a difference of 0 tonnes and 0.0 percent.
annual_source = annual[
    ["year", "lme_copper_inventory_tonnes"]
].copy()

annual_source["year"] = annual_source["year"].astype("int64")

cross_source_comparison = annual_source.merge(
    cochilco_annual,
    on="year",
    how="inner",
    validate="one_to_one"
)

cross_source_comparison["difference_tonnes"] = (
    cross_source_comparison["lme_copper_inventory_tonnes"]
    - cross_source_comparison["lme_copper_inventory_tonnes_cochilco"]
)

cross_source_comparison["difference_percent"] = (
    cross_source_comparison["difference_tonnes"]
    / cross_source_comparison[
        "lme_copper_inventory_tonnes_cochilco"
    ]
    * 100
)

display(cross_source_comparison)

,year,lme_copper_inventory_tonnes,lme_copper_inventory_tonnes_cochilco,difference_tonnes,difference_percent
0,2022,88925,88925,0,0.0


In [29]:
# Applying strict validation.
assert len(cross_source_comparison) == 1
assert cross_source_comparison["difference_tonnes"].eq(0).all()

print("Cross-source validation passed.")

Cross-source validation passed.


In [31]:
# Performing a boundary diagnostic check on the transition from the final annual observation to the first monthly observation.
# Boundary diagnostic: 2022 annual --> January 2023 monthly.

last_annual = annual_lme.loc[
    annual_lme["date"].idxmax()
]

first_monthly = monthly_lme.loc[
    monthly_lme["date"].idxmin()
]

last_annual_date = last_annual["date"]
first_monthly_date = first_monthly["date"]

last_annual_value = int(
    last_annual["lme_copper_inventory_tonnes"]
)

first_monthly_value = int(
    first_monthly["lme_copper_inventory_tonnes"]
)

absolute_change = first_monthly_value - last_annual_value

percentage_change = (
    absolute_change / last_annual_value
) * 100

boundary_diagnostic = pd.DataFrame([{
    "last_annual_date": last_annual_date,
    "last_annual_inventory_tonnes": last_annual_value,
    "first_monthly_date": first_monthly_date,
    "first_monthly_inventory_tonnes": first_monthly_value,
    "absolute_change_tonnes": absolute_change,
    "percentage_change": percentage_change
}])

display(boundary_diagnostic)

,last_annual_date,last_annual_inventory_tonnes,first_monthly_date,first_monthly_inventory_tonnes,absolute_change_tonnes,percentage_change
0,2022-12-31,88925,2023-01-31,75225,-13700,-15.406241


In [36]:
# The LME inventory series declines by 13,700 tonnes.
# (-15.406%) between December 2022 and January 2023.
# This is a diagnostic of the annual-to-monthly source transition, rather than a source discrepancy.

# Confirming dates explicitly for documentation purposes:
# Printing actual and expected dates for assertion.
expected_last_annual_date = pd.Timestamp("2022-12-31")
expected_first_monthly_date = pd.Timestamp("2023-01-31")

print("Actual final annual date:", last_annual_date)
print("Expected final annual date:", expected_last_annual_date)

print("Actual first monthly date:", first_monthly_date)
print("Expected first monthly date:", expected_first_monthly_date)

Actual final annual date: 2022-12-31 00:00:00
Expected final annual date: 2022-12-31 00:00:00
Actual first monthly date: 2023-01-31 00:00:00
Expected first monthly date: 2023-01-31 00:00:00


In [37]:
# Calculating date gaps.
days_between = (
    first_monthly_date - last_annual_date
).days

months_between = (
    (first_monthly_date.year - last_annual_date.year) * 12
    + first_monthly_date.month
    - last_annual_date.month
)

print("Days between observations:", days_between)
print("Months between observations:", months_between)

Days between observations: 31
Months between observations: 1


In [38]:
# Assertions for the dates and month transition.
assert last_annual_date == expected_last_annual_date
assert first_monthly_date == expected_first_monthly_date
assert months_between == 1

print("Boundary date validation passed.")

Boundary date validation passed.


In [39]:
# Concerning the boundary diagnostic, the results were as expected:
# The observations represent different month-ends, thus, they were not expected to be equal.
# Furthermore, the 31-day difference is recorded for documentation purposes.
# However, it should not be confused as the primary rule for datasets within the project, as month-end gaps may vary. E.g. 28, 29, 30, or 31 days.
# The more meaningful result here is that observations are exactly one month apart and correctly transition from December 2022 to January 2023.

# Context check for the changes per the boundary diagnostic, with the monthly changes within the COCHILCO series:
monthly_context = monthly_lme.sort_values("date").copy()

monthly_context["monthly_percentage_change"] = (
    monthly_context["lme_copper_inventory_tonnes"]
    .pct_change()
    * 100
)

display(
    monthly_context[
        ["date", "lme_copper_inventory_tonnes",
         "monthly_percentage_change"]
    ]
)

,date,lme_copper_inventory_tonnes,monthly_percentage_change
0,2023-01-31,75225,NaN
1,2023-02-28,64100,-14.788966
2,2023-03-31,64725,0.975039
3,2023-04-30,64550,-0.270375
4,2023-05-31,99150,53.601859
5,2023-06-30,72975,-26.399395
6,2023-07-31,68350,-6.337787
7,2023-08-31,102900,50.548647
8,2023-09-30,167825,63.095238
9,2023-10-31,176475,5.154178


In [40]:
# Context check results are as expected. Key Observations:
    # First percentage change is NaN, since January 2023 has no prior observation.
    # There are 39 percentage changes for 40 inventory observations.
    # The largest increase is ~ +63.10%.
    # The largest decline is ~ -39.53%.
    # The boundary change of -15.406 falls within the observed monthly range.

# Formally verifying the calculation.
import numpy as np

changes = monthly_context["monthly_percentage_change"].dropna()

assert len(changes) == len(monthly_context) - 1
assert np.isfinite(changes).all()

expected_changes = (
    monthly_context["lme_copper_inventory_tonnes"]
    .pct_change()
    .dropna()
    * 100
)

assert np.allclose(
    changes.to_numpy(),
    expected_changes.to_numpy()
)

print("Monthly percentage-change calculation passed.")

Monthly percentage-change calculation passed.


In [41]:
# Summarizing range for validation record.
print("Minimum monthly change:", changes.min())
print("Maximum monthly change:", changes.max())
print("Median monthly change:", changes.median())

Minimum monthly change: -39.53294412010009
Maximum monthly change: 63.095238095238095
Median monthly change: -0.13818516812529325


In [44]:
# Large movements should be treated as diagnostic observations instead of automatic erros.
# Exchange inventories may change sharply due to numerous factors, including deliveries, cancellations, warehouse movements, and market conditions.
# Check is largely sufficient. Will document the boundary diagnostic in a separate validation log.
# Additionally, both validated source segments now represent the same metric.

# Therefore, stacking into one canoncial series.
from pathlib import Path
import pandas as pd

# Preparing annual segment.
annual_canonical = annual_lme.copy()

annual_canonical["date"] = pd.to_datetime(
    annual_canonical["date"],
    errors="raise"
)

annual_canonical["lme_copper_inventory_tonnes"] = (
    pd.to_numeric(
        annual_canonical["lme_copper_inventory_tonnes"],
        errors="raise"
    )
    .astype("int64")
)

annual_canonical["observation_frequency"] = "annual"
annual_canonical["source"] = "Copper Council"
annual_canonical["source_locator"] = "Annual exchange-inventory table"

# Preparing monthly segment.
monthly_canonical = monthly_lme.copy()

monthly_canonical["date"] = pd.to_datetime(
    monthly_canonical["date"],
    errors="raise"
)

monthly_canonical["lme_copper_inventory_tonnes"] = (
    pd.to_numeric(
        monthly_canonical["lme_copper_inventory_tonnes"],
        errors="raise"
    )
    .astype("int64")
)

monthly_canonical["observation_frequency"] = "monthly"
monthly_canonical["source"] = (
    "Cochilco Table 4.1; underlying Reuters data"
)

monthly_canonical["source_locator"] = (
    "Cochilco Table 4.1 monthly LME total; "
    "monthly observation blocks"
)

# Keeping columns identical and combining the two.
canonical_lme = pd.concat(
    [
        annual_canonical[
            [
                "date",
                "lme_copper_inventory_tonnes",
                "observation_frequency",
                "source",
                "source_locator"
            ]
        ],
        monthly_canonical[
            [
                "date",
                "lme_copper_inventory_tonnes",
                "observation_frequency",
                "source",
                "source_locator"
            ]
        ]
    ],
    ignore_index=True
).sort_values("date").reset_index(drop=True)

In [45]:
# Validating combined series.
assert canonical_lme["date"].is_unique
assert canonical_lme["date"].notna().all()
assert canonical_lme[
    "lme_copper_inventory_tonnes"
].notna().all()

assert (
    canonical_lme["observation_frequency"]
    .isin(["annual", "monthly"])
    .all()
)

assert (
    canonical_lme["lme_copper_inventory_tonnes"] >= 0
).all()

print("Annual observations:",
      (canonical_lme["observation_frequency"] == "annual").sum())

print("Monthly observations:",
      (canonical_lme["observation_frequency"] == "monthly").sum())

print("Total observations:", len(canonical_lme))
print("Date range:",
      canonical_lme["date"].min(),
      "to",
      canonical_lme["date"].max())

Annual observations: 23
Monthly observations: 40
Total observations: 63
Date range: 2000-12-31 00:00:00 to 2026-04-30 00:00:00


In [46]:
# Counts are as expected.
# Inspecting transition.
display(
    canonical_lme[
        (canonical_lme["date"] >= "2022-01-01") &
        (canonical_lme["date"] <= "2023-03-31")
    ]
)

,date,lme_copper_inventory_tonnes,observation_frequency,source,source_locator
22,2022-12-31,88925,annual,Copper Council,Annual exchange-inventory table
23,2023-01-31,75225,monthly,Cochilco Table 4.1; underlying Reuters data,Cochilco Table 4.1 monthly LME total; monthly observation blocks
24,2023-02-28,64100,monthly,Cochilco Table 4.1; underlying Reuters data,Cochilco Table 4.1 monthly LME total; monthly observation blocks
25,2023-03-31,64725,monthly,Cochilco Table 4.1; underlying Reuters data,Cochilco Table 4.1 monthly LME total; monthly observation blocks


In [48]:
# Underlying values are as expected; frequency and source transition are clearly documented.

# Saving the canonical series.
output_path = Path(
    r"..\data\processed\lme_copper_inventory_canonical_clean.csv"
)

canonical_lme.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ..\data\processed\lme_copper_inventory_canonical_clean.csv
